1. Baseline Model - Linear Regression
2. CatBoost
3. Random Forest
4. XGBoost
5. 

### Setup

In [1]:
#Imports
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import re

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier, Pool
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

In [2]:
csv_path = Path.cwd().parent / "data" / "all_fights.csv"
data_frame = pd.read_csv(csv_path)

data_frame = pd.read_csv(csv_path)
data_frame = data_frame[~data_frame["blue_stance"].isin(["Switch ", "Open Stance", "Unknown"])]
data_frame = data_frame[~data_frame["red_stance"].isin(["Switch ", "Open Stance", "Unknown"])]

# Drop Catch Weight from weight_class (72 entries)
data_frame = data_frame[~data_frame["weight_class"].isin(["Catch Weight"])]

# Drop reach diff over 40 (1 entry)
data_frame = data_frame[(data_frame["reach_diff"] > -40) & (data_frame["reach_diff"] < 40)]

# Drop round_diff outlier (1 entry)
data_frame = data_frame[(data_frame["rounds_diff"] > -100) & (data_frame["rounds_diff"] < 100)]

# Convert red_winner from string to bool
data_frame["red_winner"] = (data_frame["red_winner"].astype(str).str.lower() == "t").astype(int)
# Convert title bout from string to bool'
data_frame["title_bout"] = (data_frame["title_bout"].astype(str).str.lower() == "t").astype(int)
# ============

# Get all diff features
diff_features = data_frame.filter(regex=r'_diff')
diff_features_no_odds = [f for f in diff_features if f != 'odds_diff']

extra_features = [
    "gender", "weight_class", "red_stance", "blue_stance", "red_age", 
    "blue_age", "b_match_wc_rank", "r_match_wc_rank"
]

#Target
y = data_frame['red_winner']

# Temporal Split

DATE = "2023-06-01"
train_mask = data_frame['fight_date'] <= DATE
test_mask = data_frame['fight_date'] > DATE

y_train = y[train_mask]
y_test = y[test_mask]

# Build feature dataframe with all extras + all diffs (including odds)
all_features_df = pd.concat([data_frame[extra_features], data_frame.filter(regex='_diff')], axis=1)

# Replace instances of rank 20 (unranked) to 16 so ranking distribution is uniform 
worst_rank = max(all_features_df[["b_match_wc_rank", "r_match_wc_rank"]].max().max(), 16) + 1
all_features_df["b_match_wc_rank"] = all_features_df["b_match_wc_rank"].fillna(worst_rank)
all_features_df["r_match_wc_rank"] = all_features_df["r_match_wc_rank"].fillna(worst_rank)

# Age gap is more/less impactful depending on where it sits. A 25 v 28 age gap is less significant 
# than it being 32 v 35 and raw age_diff doesn't see that.
all_features_df["avg_age"] = (all_features_df.red_age + all_features_df.blue_age) / 2
all_features_df["avg_age_c"] = all_features_df["avg_age"] - all_features_df["avg_age"].mean()  # center it

# Create a new column that captures the interaction: how much age_diff's
# effect should be amplified or dampened based on how old the pair is on average
all_features_df["age_diff_x_avg"] = all_features_df["age_diff"] * all_features_df["avg_age_c"]

# Drop raw ages since we already have the affect
all_features_df = all_features_df.drop(columns=["red_age", "blue_age"])

# Split into with/without odds
cols_with_odds = [c for c in all_features_df.columns]
cols_no_odds = [c for c in cols_with_odds if c != 'odds_diff']

# Train/test with odds
X_train_odds = all_features_df.loc[train_mask, cols_with_odds]
X_test_odds = all_features_df.loc[test_mask, cols_with_odds]

#Train/test without odds
X_train_no_odds = all_features_df.loc[train_mask, cols_no_odds]
X_test_no_odds = all_features_df.loc[test_mask, cols_no_odds]



## Testing models
### Baseline Model - Logistic Regression

In [3]:
# Logistic Regression
# Odds included
X_test_dummy_odds = pd.get_dummies(X_test_odds, columns=['gender', 'weight_class', 'red_stance', 'blue_stance'])
X_train_dummy_odds = pd.get_dummies(X_train_odds, columns=['gender', 'weight_class', 'red_stance', 'blue_stance'])
lr = LogisticRegression(max_iter=10000)
lr.fit(X_train_dummy_odds, y_train)
preds_odds = lr.predict(X_test_dummy_odds)
print('=== With Odds Diff ===')
print(accuracy_score(y_test, preds_odds))
print(classification_report(y_test, preds_odds))
# Odds not included
X_test_dummy_no_odds = pd.get_dummies(X_test_no_odds, columns=['gender', 'weight_class', 'red_stance', 'blue_stance'])
X_train_dummy_no_odds = pd.get_dummies(X_train_no_odds, columns=['gender', 'weight_class', 'red_stance', 'blue_stance'])
lr = LogisticRegression(max_iter=10000)
lr.fit(X_train_dummy_no_odds, y_train)
preds_no_odds = lr.predict(X_test_dummy_no_odds)
print('=== Without Odds Diff ===')
print(accuracy_score(y_test, preds_no_odds))
print(classification_report(y_test, preds_no_odds))

=== With Odds Diff ===
0.6996699669966997
              precision    recall  f1-score   support

           0       0.69      0.59      0.64       673
           1       0.71      0.79      0.74       842

    accuracy                           0.70      1515
   macro avg       0.70      0.69      0.69      1515
weighted avg       0.70      0.70      0.70      1515

=== Without Odds Diff ===
0.6026402640264027
              precision    recall  f1-score   support

           0       0.59      0.36      0.45       673
           1       0.61      0.80      0.69       842

    accuracy                           0.60      1515
   macro avg       0.60      0.58      0.57      1515
weighted avg       0.60      0.60      0.58      1515



### Catboost - odds_diff included

In [4]:
# your categorical columns (everything that's object/category dtype)
cat_features = ["gender", "weight_class", "red_stance", "blue_stance"]

X_train_cat = X_train_odds.copy()
X_test_cat = X_test_odds.copy()

# CatBoost needs cats as string, and no NaNs in cat columns (fill or drop first)
for col in cat_features:
    X_train_cat[col] = X_train_cat[col].astype(str)
    X_test_cat[col] = X_test_cat[col].astype(str)

train_pool = Pool(X_train_cat, y_train, cat_features=cat_features)
test_pool = Pool(X_test_cat, y_test, cat_features=cat_features)

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    eval_metric="Accuracy",
    random_seed=26,
    verbose=100  # prints progress every 100 iterations
)

model.fit(train_pool, eval_set=test_pool, early_stopping_rounds=50)

preds = model.predict(X_test_cat)
print(accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

# feature importance
importances = model.get_feature_importance(train_pool, prettified=True)
print(importances)

0:	learn: 0.6528415	test: 0.7042904	best: 0.7042904 (0)	total: 60.9ms	remaining: 30.4s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.704950495
bestIteration = 7

Shrink model to first 8 iterations.
0.7049504950495049
              precision    recall  f1-score   support

           0       0.69      0.62      0.65       673
           1       0.72      0.78      0.75       842

    accuracy                           0.70      1515
   macro avg       0.70      0.70      0.70      1515
weighted avg       0.70      0.70      0.70      1515

                 Feature Id  Importances
0                 odds_diff    88.487540
1                   td_diff     2.611120
2              sig_str_diff     1.900394
3                reach_diff     1.481786
4                  age_diff     1.119471
5                   ko_diff     1.014479
6           win_streak_diff     0.842662
7           b_match_wc_rank     0.812696
8              sub_att_diff     0.544884
9           r_match_wc_r

### Catboost - odds_diff not included

In [5]:
# your categorical columns (everything that's object/category dtype)
cat_features = ["gender", "weight_class", "red_stance", "blue_stance"]

X_train_cat_no_odds = X_train_no_odds.copy()
X_test_cat_no_odds = X_test_no_odds.copy()

# CatBoost needs cats as string, and no NaNs in cat columns (fill or drop first)
for col in cat_features:
    X_train_cat_no_odds[col] = X_train_cat_no_odds[col].astype(str)
    X_test_cat_no_odds[col] = X_test_cat_no_odds[col].astype(str)

train_pool = Pool(X_train_cat_no_odds, y_train, cat_features=cat_features)
test_pool = Pool(X_test_cat_no_odds, y_test, cat_features=cat_features)

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    eval_metric="Accuracy",
    random_seed=26,
    verbose=100  # prints progress every 100 iterations
)

model.fit(train_pool, eval_set=test_pool, early_stopping_rounds=50)

preds = model.predict(X_test_cat_no_odds)
print(accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

# feature importance
importances = model.get_feature_importance(train_pool, prettified=True)
print(importances)

0:	learn: 0.5887752	test: 0.5524752	best: 0.5524752 (0)	total: 7.24ms	remaining: 3.61s
100:	learn: 0.6561948	test: 0.6033003	best: 0.6085809 (50)	total: 367ms	remaining: 1.45s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.6085808581
bestIteration = 50

Shrink model to first 51 iterations.
0.6085808580858085
              precision    recall  f1-score   support

           0       0.64      0.27      0.38       673
           1       0.60      0.88      0.71       842

    accuracy                           0.61      1515
   macro avg       0.62      0.57      0.55      1515
weighted avg       0.62      0.61      0.56      1515

                 Feature Id  Importances
0                  age_diff    26.591779
1                   td_diff    10.851530
2              sig_str_diff     9.258144
3               losses_diff     6.110633
4           win_streak_diff     6.007676
5                red_stance     3.814879
6                reach_diff     3.777530
7             

### Random Forest - odds_diff included

In [11]:
# Random Forest
X_train_forest = X_train_odds.copy()
X_test_forest = X_test_odds.copy()

X_train_dummy_odds = pd.get_dummies(X_train_forest, columns=['gender', 'weight_class', 'red_stance', 'blue_stance'])
X_test_dummy_odds = pd.get_dummies(X_test_forest, columns=['gender', 'weight_class', 'red_stance', 'blue_stance'])

param_grid = {
    'n_estimators' : [100, 200, 500],
    'max_depth': [3, 5, 10, None],
    'min_samples_leaf': [1, 5, 10, 20]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=26),
    param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1
)
grid_search.fit(X_train_dummy_odds, y_train)

print('=== Best params ===')
print(grid_search.best_params_)
print('=== Best score ===')
print(grid_search.best_score_)

best_rf = grid_search.best_estimator_
preds = best_rf.predict(X_test_dummy_odds)
print('=== Scores ===')
print(accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

print('=== With Odds ===')
importances = pd.Series(best_rf.feature_importances_, index=X_train_dummy_odds.columns)
print(importances.sort_values(ascending=False).head(10))

Fitting 5 folds for each of 48 candidates, totalling 240 fits
=== Best params ===
{'max_depth': None, 'min_samples_leaf': 20, 'n_estimators': 500}
=== Best score ===
0.6565451089722935
=== Scores ===
0.6877887788778878
              precision    recall  f1-score   support

           0       0.69      0.54      0.61       673
           1       0.69      0.80      0.74       842

    accuracy                           0.69      1515
   macro avg       0.69      0.67      0.67      1515
weighted avg       0.69      0.69      0.68      1515

=== With Odds ===
odds_diff         0.377560
age_diff          0.077100
sig_str_diff      0.062300
td_diff           0.061783
age_diff_x_avg    0.042435
sub_att_diff      0.038430
rounds_diff       0.034925
losses_diff       0.032347
reach_diff        0.028847
avg_age           0.026479
dtype: float64


### Random Forest - odds_diff not included

In [10]:
X_train_forest_no_odds = X_train_no_odds.copy()
X_test_forest_no_odds = X_test_no_odds.copy()

X_train_dummy_no_odds = pd.get_dummies(X_train_forest_no_odds, columns=['gender', 'weight_class', 'red_stance', 'blue_stance'])
X_test_dummy_no_odds = pd.get_dummies(X_test_forest_no_odds, columns=['gender', 'weight_class', 'red_stance', 'blue_stance'])

param_grid = {
    'n_estimators' : [100, 200, 500],
    'max_depth': [3, 5, 10, None],
    'min_samples_leaf': [1, 5, 10, 20]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=26),
    param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1
)
grid_search.fit(X_train_dummy_no_odds, y_train)

print('=== Best params ===')
print(grid_search.best_params_)
print('=== Best score ===')
print(grid_search.best_score_)

best_rf = grid_search.best_estimator_
preds = best_rf.predict(X_test_dummy_no_odds)
print('=== Scores ===')
print(accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

print('=== Importances ===')
importances = pd.Series(best_rf.feature_importances_, index=X_train_dummy_no_odds.columns)
print(importances.sort_values(ascending=False).head(10))

Fitting 5 folds for each of 48 candidates, totalling 240 fits
=== Best params ===
{'max_depth': 10, 'min_samples_leaf': 5, 'n_estimators': 200}
=== Best score ===
0.6094218498749243
=== Scores ===
0.6072607260726073
              precision    recall  f1-score   support

           0       0.62      0.30      0.40       673
           1       0.60      0.85      0.71       842

    accuracy                           0.61      1515
   macro avg       0.61      0.58      0.56      1515
weighted avg       0.61      0.61      0.57      1515

=== Importances ===
age_diff          0.119036
td_diff           0.094281
sig_str_diff      0.085617
age_diff_x_avg    0.060454
sub_att_diff      0.056079
rounds_diff       0.052886
losses_diff       0.051479
reach_diff        0.040512
avg_age_c         0.039430
avg_age           0.037542
dtype: float64


### XGBoost - odds_diff included

In [ ]:
categorical_features = ["red_stance", "blue_stance", "weight_class", "gender"]
numeric_features = [
    "age_diff", "td_diff", "sig_str_diff", "reach_diff", "losses_diff",
    "sub_att_diff", "rank_diff",
    "win_streak_diff", "longest_win_streak_diff", "lose_streak_diff",
    "title_bout_diff", "wins_diff", "rounds_diff", "height_diff",
    "ko_diff", "submission_diff", "b_match_wc_rank", "r_match_wc_rank", 
    "avg_age", "avg_age_c", "age_diff_x_avg"
]

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

pipeline_xgb = Pipeline([
    ("preprocess", preprocessor),
    ("clf", XGBClassifier(
        eval_metric="logloss",
        random_state=42
    ))
])

param_grid = {
    "clf__n_estimators": [100, 300],
    "clf__max_depth": [3, 6],
    "clf__learning_rate": [0.01, 0.1],
    "clf__subsample": [0.8, 1.0],
    "clf__colsample_bytree": [0.8, 1.0]
}

grid_search = GridSearchCV(
    pipeline_xgb,
    param_grid,
    cv=5,
    scoring="roc_auc",   # accuracy alone is weak if red_winner is imbalanced — see note below
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train_odds, y_train)

print("Best parameters:")
print(grid_search.best_params_)
print()
print("Best cross-validation ROC-AUC:")
print(grid_search.best_score_)

best_index = grid_search.best_index_
print("Training score:", grid_search.cv_results_["mean_train_score"][best_index])
print("Validation score:", grid_search.cv_results_["mean_test_score"][best_index])

print("================== Feature Importance ==================")
best_model = grid_search.best_estimator_
xgb_clf = best_model.named_steps["clf"]

# Get feature names after one-hot encoding
feature_names = best_model.named_steps["preprocess"].get_feature_names_out()

importances = xgb_clf.feature_importances_
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print(importance_df.head(15))

plt.figure(figsize=(8, 6))
sns.barplot(data=importance_df.head(15), x="importance", y="feature")
plt.title("Top 15 feature importances (XGBoost)")
plt.tight_layout()
plt.show()

best_xgb = grid_search.best_estimator_
preds = best_xgb.predict(X_test_odds)
print('=== Test Set ===')
print(accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

ModuleNotFoundError: No module named 'xgboost'

### MORE THINGS

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV

from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier


x_train = x_training_df


# --- Feature groups ---
categorical_features = ["red_stance", "blue_stance", "weight_class", "gender",
                         "red_age_bucket", "blue_age_bucket"]
numeric_features = [
    "age_diff", "td_diff", "sig_str_diff", "reach_diff", "losses_diff",
    "sub_att_diff", "rank_diff", "win_streak_diff", "longest_win_streak_diff",
    "lose_streak_diff", "title_bout_diff", "wins_diff", "rounds_diff",
    "height_diff", "ko_diff", "submission_diff"
]

# --- Preprocessors ---
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

preprocessor_scaled = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

# --- Pipelines ---
pipeline_xgb = Pipeline([
    ("preprocess", preprocessor),
    ("clf", XGBClassifier(eval_metric="logloss", random_state=42))
])

pipeline_lr = Pipeline([
    ("preprocess", preprocessor_scaled),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

pipeline_rf = Pipeline([
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(random_state=42))
])

pipeline_knn = Pipeline([
    ("preprocess", preprocessor_scaled),
    ("clf", KNeighborsClassifier())
])

# --- Param grids ---
param_grid_xgb = {
    "clf__n_estimators": [100, 300],
    "clf__max_depth": [3, 6],
    "clf__learning_rate": [0.01, 0.1],
    "clf__subsample": [0.8, 1.0],
    "clf__colsample_bytree": [0.8, 1.0]
}

param_grid_lr = {
    "clf__C": [0.01, 0.1, 1.0, 10.0],
    "clf__penalty": ["l1", "l2"],
    "clf__solver": ["liblinear"]
}

param_grid_rf = {
    "clf__n_estimators": [100, 300],
    "clf__max_depth": [None, 10, 20],
    "clf__min_samples_leaf": [1, 5],
    "clf__max_features": ["sqrt", "log2"]
}

param_grid_knn = {
    "clf__n_neighbors": [5, 11, 21, 31],
    "clf__weights": ["uniform", "distance"],
    "clf__p": [1, 2]
}

# --- Grid searches ---
grid_search_xgb = GridSearchCV(pipeline_xgb, param_grid_xgb, cv=5,
                                scoring="roc_auc", n_jobs=-1, return_train_score=True)
grid_search_lr = GridSearchCV(pipeline_lr, param_grid_lr, cv=5,
                               scoring="roc_auc", n_jobs=-1, return_train_score=True)
grid_search_rf = GridSearchCV(pipeline_rf, param_grid_rf, cv=5,
                               scoring="roc_auc", n_jobs=-1, return_train_score=True)
grid_search_knn = GridSearchCV(pipeline_knn, param_grid_knn, cv=5,
                                scoring="roc_auc", n_jobs=-1, return_train_score=True)

grid_search_xgb.fit(x_train, y_train)
print("XGBoost done —", grid_search_xgb.best_params_, grid_search_xgb.best_score_)

grid_search_lr.fit(x_train, y_train)
print("Logistic Regression done —", grid_search_lr.best_params_, grid_search_lr.best_score_)

grid_search_rf.fit(x_train, y_train)
print("Random Forest done —", grid_search_rf.best_params_, grid_search_rf.best_score_)

grid_search_knn.fit(x_train, y_train)
print("kNN done —", grid_search_knn.best_params_, grid_search_knn.best_score_)

# --- Summary table ---
results_summary = pd.DataFrame({
    "model": ["XGBoost", "Logistic Regression", "Random Forest", "kNN"],
    "best_cv_roc_auc": [
        grid_search_xgb.best_score_,
        grid_search_lr.best_score_,
        grid_search_rf.best_score_,
        grid_search_knn.best_score_
    ]
}).sort_values("best_cv_roc_auc", ascending=False)

print()
print(results_summary)

XGBoost done — {'clf__colsample_bytree': 0.8, 'clf__learning_rate': 0.01, 'clf__max_depth': 3, 'clf__n_estimators': 300, 'clf__subsample': 0.8} 0.6147488856122709


C:\Users\tumik\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\tumik\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Logistic Regression done — {'clf__C': 0.1, 'clf__penalty': 'l1', 'clf__solver': 'liblinear'} 0.6228544545802496
Random Forest done — {'clf__max_depth': 10, 'clf__max_features': 'log2', 'clf__min_samples_leaf': 5, 'clf__n_estimators': 300} 0.6163616736881987
kNN done — {'clf__n_neighbors': 31, 'clf__p': 2, 'clf__weights': 'distance'} 0.5780621550719072

                 model  best_cv_roc_auc
1  Logistic Regression         0.622854
2        Random Forest         0.616362
0              XGBoost         0.614749
3                  kNN         0.578062


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import GridSearchCV
from catboost import CatBoostClassifier

# --- Feature groups ---
categorical_features = ["gender", "weight_class", "red_stance", "blue_stance",
                         "red_age_bucket", "blue_age_bucket"]
numeric_features = [
    "age_diff", "td_diff", "sig_str_diff", "reach_diff", "losses_diff",
    "sub_att_diff", "rank_diff", "win_streak_diff", "longest_win_streak_diff",
    "lose_streak_diff", "title_bout_diff", "wins_diff", "rounds_diff",
    "height_diff", "ko_diff", "submission_diff"
]

# --- Preprocessing: impute only — CatBoost handles cats and scale itself ---
preprocessor = ColumnTransformer([
    ("cat", SimpleImputer(strategy="constant", fill_value="missing"), categorical_features),
    ("num", SimpleImputer(strategy="median"), numeric_features)
], verbose_feature_names_out=False).set_output(transform="pandas")

# force categorical columns back to string after imputation (SimpleImputer can upcast dtypes)
def cast_cats_to_str(df):
    df = df.copy()
    df[categorical_features] = df[categorical_features].astype(str)
    return df

cast_step = FunctionTransformer(cast_cats_to_str)

# --- Pipeline (no cat_features in the constructor — passed via fit() instead) ---
pipeline_cb = Pipeline([
    ("preprocess", preprocessor),
    ("cast", cast_step),
    ("clf", CatBoostClassifier(
        random_seed=26,
        verbose=0
    ))
])

# --- Param grid ---
""" param_grid_cb = {
    "clf__iterations": [300, 500],
    "clf__learning_rate": [0.03, 0.05, 0.1],
    "clf__depth": [4, 6, 8],
    "clf__l2_leaf_reg": [1, 3, 5]
} """

param_grid_cb = {
    "clf__iterations": [300, 500],
    "clf__learning_rate": [0.03, 0.05],
    "clf__depth": [4, 6],
    "clf__l2_leaf_reg": [1, 3]
}
# Hyper parameters for CatBoostClassifier


# --- Grid search ---
grid_search_cb = GridSearchCV(
    pipeline_cb, param_grid_cb,
    cv=5, scoring="roc_auc", n_jobs=-1, return_train_score=True
)


# cat_features passed here via clf__ prefix, forwarded to CatBoostClassifier.fit() on every fold
grid_search_cb.fit(x_train, y_train, clf__cat_features=categorical_features)

print("Best parameters:")
print(grid_search_cb.best_params_)
print()
print("Best cross-validation ROC-AUC:")
print(grid_search_cb.best_score_)

best_index = grid_search_cb.best_index_
print("Training score:", grid_search_cb.cv_results_["mean_train_score"][best_index])
print("Validation score:", grid_search_cb.cv_results_["mean_test_score"][best_index])